# 08 Dataset Comparison

This notebook compares the original one-year Online Retail workbook with the expanded two-year Online Retail II CSV.

It is designed to answer two questions:

- Is the expanded dataset independent, or does it contain the current project baseline?
- Do the main retention, concentration, ML, and CRM priority readouts remain directionally consistent on the longer history?

In [ ]:
from pathlib import Path

import pandas as pd

RAW_V1_PATH = Path("../data/raw/Online Retail.xlsx")
RAW_V2_PATH = Path("../data/raw/online_retail_II.csv")
PROCESSED_BASE = Path("../data/processed")

DATASETS = {
    "v1_current_one_year": PROCESSED_BASE / "v1_online_retail",
    "v2_expanded_two_year": PROCESSED_BASE / "v2_online_retail_ii",
}

for path in [RAW_V1_PATH, RAW_V2_PATH, *DATASETS.values()]:
    if not path.exists():
        raise FileNotFoundError(f"Required comparison input not found: {path}")

## Raw Dataset Overlap

In [ ]:
v1_raw = pd.read_excel(RAW_V1_PATH)
v2_raw = pd.read_csv(RAW_V2_PATH).rename(
    columns={"Invoice": "InvoiceNo", "Price": "UnitPrice", "Customer ID": "CustomerID"}
)

comparison_columns = [
    "InvoiceNo",
    "StockCode",
    "Description",
    "Quantity",
    "InvoiceDate",
    "UnitPrice",
    "CustomerID",
    "Country",
]

for raw in [v1_raw, v2_raw]:
    raw["InvoiceNo"] = raw["InvoiceNo"].astype(str)
    raw["StockCode"] = raw["StockCode"].astype(str)
    raw["InvoiceDate"] = pd.to_datetime(raw["InvoiceDate"])
    raw["Description"] = raw["Description"].astype("string").fillna("<NA>")
    raw["CustomerID"] = raw["CustomerID"].fillna(-1).astype(float).astype(int)

v1_counts = v1_raw.groupby(comparison_columns, dropna=False).size().rename("v1_count").reset_index()
v2_counts = v2_raw.groupby(comparison_columns, dropna=False).size().rename("v2_count").reset_index()
overlap = v1_counts.merge(v2_counts, on=comparison_columns, how="left")
overlap["v2_count"] = overlap["v2_count"].fillna(0).astype(int)
overlap["matched_rows"] = overlap[["v1_count", "v2_count"]].min(axis=1)

raw_overlap_summary = pd.DataFrame(
    {
        "metric": [
            "v1_rows",
            "v2_rows",
            "v1_rows_found_in_v2",
            "v1_rows_found_in_v2_pct",
            "v1_invoice_overlap_pct",
            "v1_customer_overlap_pct",
            "v1_stock_overlap_pct",
        ],
        "value": [
            len(v1_raw),
            len(v2_raw),
            overlap["matched_rows"].sum(),
            overlap["matched_rows"].sum() / len(v1_raw),
            len(set(v1_raw["InvoiceNo"]) & set(v2_raw["InvoiceNo"])) / v1_raw["InvoiceNo"].nunique(),
            len(set(v1_raw.loc[v1_raw["CustomerID"] != -1, "CustomerID"]) & set(v2_raw.loc[v2_raw["CustomerID"] != -1, "CustomerID"])) / v1_raw.loc[v1_raw["CustomerID"] != -1, "CustomerID"].nunique(),
            len(set(v1_raw["StockCode"]) & set(v2_raw["StockCode"])) / v1_raw["StockCode"].nunique(),
        ],
    }
)
raw_overlap_summary

## Analytical Output Comparison

In [ ]:
def audit_value(path, metric):
    audit = pd.read_csv(path / "data_audit_summary.csv").set_index("metric")["value"]
    return float(audit.loc[metric])


def lifecycle_share(path, metric):
    lifecycle = pd.read_csv(path / "customer_lifecycle_revenue_split.csv").set_index("metric")
    return float(lifecycle.loc[metric, "share_of_identifiable_revenue"])


def top_decile_share(path):
    deciles = pd.read_csv(path / "customer_revenue_deciles.csv")
    return float(deciles.loc[deciles["customer_decile"].eq("Top 10%"), "revenue_pct"].iloc[0])


def order_table(path):
    transactions = pd.read_parquet(path / "clean_transactions.parquet")
    transactions.columns = transactions.columns.str.strip().str.lower().str.replace(" ", "_")
    transactions = transactions.rename(
        columns={
            "invoiceno": "invoice_no",
            "stockcode": "stock_code",
            "invoicedate": "invoice_date",
            "customerid": "customer_id",
        }
    )
    transactions["invoice_date"] = pd.to_datetime(transactions["invoice_date"])
    valid = transactions[transactions["is_valid_order_line"]].copy()
    orders = (
        valid.groupby(["customer_id", "invoice_no"], as_index=False)
        .agg(order_date=("invoice_date", "min"), order_revenue=("analytical_revenue", "sum"))
    )
    orders = orders[orders["order_revenue"] > 0].sort_values(["customer_id", "order_date", "invoice_no"])
    orders["order_rank"] = orders.groupby("customer_id").cumcount() + 1
    return orders


def second_purchase_conversion(path):
    customers = pd.read_parquet(path / "customer_metrics.parquet")
    return float((customers["total_orders"] > 1).mean())


def median_days_to_second_purchase(path):
    orders = order_table(path)
    first = orders[orders["order_rank"].eq(1)][["customer_id", "order_date"]].rename(columns={"order_date": "first_order_date"})
    second = orders[orders["order_rank"].eq(2)][["customer_id", "order_date"]].rename(columns={"order_date": "second_order_date"})
    second_days = first.merge(second, on="customer_id")
    return float((second_days["second_order_date"] - second_days["first_order_date"]).dt.days.median())


def selected_model(path, file_name):
    metrics = pd.read_csv(path / file_name)
    return metrics.sort_values(["precision_top_20pct", "roc_auc"], ascending=False).iloc[0]


comparison_rows = []
for dataset, path in DATASETS.items():
    second_purchase = selected_model(path, "second_purchase_model_metrics.csv")
    lapse = selected_model(path, "lapse_risk_model_metrics.csv")
    priority = pd.read_csv(path / "crm_priority_summary.csv")
    p1 = priority.loc[priority["crm_priority_group"].eq("P1 High-Value Lapse Prevention")].iloc[0]

    comparison_rows.append(
        {
            "dataset": dataset,
            "total_rows": audit_value(path, "total_rows"),
            "missing_customer_pct": audit_value(path, "missing_customer_pct"),
            "identifiable_customer_revenue": audit_value(path, "identifiable_customer_revenue"),
            "identifiable_revenue_pct": audit_value(path, "identifiable_revenue_pct"),
            "valid_customers": audit_value(path, "valid_customers"),
            "valid_orders": audit_value(path, "valid_orders"),
            "second_purchase_conversion": second_purchase_conversion(path),
            "median_days_to_second_purchase": median_days_to_second_purchase(path),
            "repeat_order_revenue_share": lifecycle_share(path, "repeat_order_revenue"),
            "repeat_customer_revenue_share": lifecycle_share(path, "repeat_customer_revenue"),
            "top_10_revenue_share": top_decile_share(path),
            "second_purchase_model": second_purchase["model"],
            "second_purchase_roc_auc": second_purchase["roc_auc"],
            "second_purchase_top20_precision": second_purchase["precision_top_20pct"],
            "lapse_model": lapse["model"],
            "lapse_roc_auc": lapse["roc_auc"],
            "lapse_top20_precision": lapse["precision_top_20pct"],
            "p1_customers": p1["customers"],
            "p1_revenue_share": p1["revenue_pct"],
        }
    )

dataset_comparison = pd.DataFrame(comparison_rows)
dataset_comparison

## Interpretation

The expanded v2 dataset contains the full v1 baseline and adds the prior year of trading history. Use the comparison as a robustness and longer-history test, not as a fully independent validation sample.